# Olist E-Commerce Data Cleaning and Preparation

## 1. Introduction

This notebook focuses on cleaning and preparing the Olist Brazilian E-Commerce dataset.

The objectives are:
- Identify and handle missing values.
- Investigate duplicate records.
- Correct data types.
- Validate numerical values.
- Check data consistency.
- Export cleaned datasets for further analysis.

The original raw datasets will remain unchanged. All processed datasets will be stored in the data/processed directory.

## 2. Importing Required Libraries

In [16]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

Libraries imported successfully!


## 3. Loading Original Datasets

In [17]:
# Define dataset paths

RAW_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")

# Create processed directory if it does not exist

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Load all nine original datasets

customers = pd.read_csv(RAW_PATH / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_PATH / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(RAW_PATH / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(RAW_PATH / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(RAW_PATH / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(RAW_PATH / "olist_orders_dataset.csv")
products = pd.read_csv(RAW_PATH / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_PATH / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    RAW_PATH / "product_category_name_translation.csv"
)

datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation
}

print("All nine original datasets loaded successfully!")
print("Processed folder:", PROCESSED_PATH.resolve())

All nine original datasets loaded successfully!
Processed folder: C:\Users\romyd\olist-ecommerce-analytics\data\processed


## 4. Initial Data Quality Assessment

Review missing values and duplicate records before applying cleaning operations.

In [18]:
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing Values": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum()
    })

quality_df = pd.DataFrame(quality_summary)

display(quality_df)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,customers,99441,5,0,0
1,geolocation,1000163,5,0,261831
2,order_items,112650,7,0,0
3,order_payments,103886,5,0,0
4,order_reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,2448,0
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


## 5. Handling Missing Values

Missing values are handled according to their business meaning.

- Review comments: replace missing text with empty strings for easier text processing.
- Order timestamps: retain missing values because they may represent orders that were not delivered or approved.
- Product attributes: retain missing values rather than introducing inaccurate information.

In [19]:
# Preserve missing-value indicators before cleaning

orders["delivery_date_missing"] = (
    orders["order_delivered_customer_date"].isna()
)

orders["approval_date_missing"] = (
    orders["order_approved_at"].isna()
)

products["category_missing"] = (
    products["product_category_name"].isna()
)

# Clean review text columns

order_reviews["review_comment_title"] = (
    order_reviews["review_comment_title"].fillna("")
)

order_reviews["review_comment_message"] = (
    order_reviews["review_comment_message"].fillna("")
)

print("Missing-value handling completed!")

Missing-value handling completed!


## 6. Duplicate Record Handling

The Geolocation dataset contains completely duplicated records. These exact duplicates are removed while retaining distinct geographical records.

In [20]:
# Check duplicates before removal

geo_duplicates_before = geolocation.duplicated().sum()

# Remove exact duplicate rows

geolocation = geolocation.drop_duplicates().reset_index(drop=True)

# Check duplicates after removal

geo_duplicates_after = geolocation.duplicated().sum()

print("Duplicates before cleaning:", geo_duplicates_before)
print("Duplicates after cleaning:", geo_duplicates_after)
print("Geolocation records after cleaning:", len(geolocation))

Duplicates before cleaning: 261831
Duplicates after cleaning: 0
Geolocation records after cleaning: 738332


## 7. Date and Time Data Type Conversion

Convert order timestamp columns into datetime format to support time-based analysis.

In [21]:
# Convert order timestamp columns

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

print("Order date columns converted successfully!")

display(orders[date_columns].dtypes)

Order date columns converted successfully!


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 8. Numerical Data Validation

Check for invalid negative prices, freight charges, payment values, and product weights.

In [22]:
# Count invalid numerical values

validation_results = {
    "Negative product prices": (order_items["price"] < 0).sum(),
    "Negative freight values": (order_items["freight_value"] < 0).sum(),
    "Negative payment values": (order_payments["payment_value"] < 0).sum(),
    "Negative product weights": (products["product_weight_g"] < 0).sum()
}

validation_df = pd.DataFrame(
    list(validation_results.items()),
    columns=["Validation Check", "Invalid Records"]
)

display(validation_df)

,Validation Check,Invalid Records
0,Negative product prices,0
1,Negative freight values,0
2,Negative payment values,0
3,Negative product weights,0


## 9. Primary Key Validation

Check whether important identifier columns contain missing or duplicate values.

In [23]:
# Primary key validation

primary_key_checks = {
    "Customers": (customers, "customer_id"),
    "Orders": (orders, "order_id"),
    "Products": (products, "product_id"),
    "Sellers": (sellers, "seller_id"),
    "Order Items": (order_items, ["order_id", "order_item_id"]),
    "Order Payments": (order_payments, ["order_id", "payment_sequential"]),
    "Category Translation": (category_translation, "product_category_name")
}

results = []

for name, (df, key) in primary_key_checks.items():
    results.append({
        "Dataset": name,
        "Missing Key Values": df[key].isnull().any(axis=1).sum()
            if isinstance(key, list) else df[key].isnull().sum(),
        "Duplicate Key Records": df.duplicated(subset=key).sum()
    })

primary_key_df = pd.DataFrame(results)
display(primary_key_df)

,Dataset,Missing Key Values,Duplicate Key Records
0,Customers,0,0
1,Orders,0,0
2,Products,0,0
3,Sellers,0,0
4,Order Items,0,0
5,Order Payments,0,0
6,Category Translation,0,0


## 10. Referential Integrity Validation

Verify that order items reference existing orders, customers, products, and sellers.

In [24]:
relationship_checks = {
    "Order Items without Orders": (
        ~order_items["order_id"].isin(orders["order_id"])
    ).sum(),

    "Orders without Customers": (
        ~orders["customer_id"].isin(customers["customer_id"])
    ).sum(),

    "Order Items without Products": (
        ~order_items["product_id"].isin(products["product_id"])
    ).sum(),

    "Order Items without Sellers": (
        ~order_items["seller_id"].isin(sellers["seller_id"])
    ).sum()
}

relationship_df = pd.DataFrame(
    list(relationship_checks.items()),
    columns=["Relationship Check", "Unmatched Records"]
)

display(relationship_df)

,Relationship Check,Unmatched Records
0,Order Items without Orders,0
1,Orders without Customers,0
2,Order Items without Products,0
3,Order Items without Sellers,0


## 11. Data Cleaning Summary

The following cleaning and validation operations were completed:

1. Loaded all nine original Olist datasets.
2. Replaced missing review titles and messages with empty strings.
3. Preserved missing order timestamps and product attributes.
4. Removed 261,831 exact duplicate records from the Geolocation dataset.
5. Converted order timestamp columns into datetime format.
6. Validated product prices, freight charges, payment values, and product weights.
7. Verified primary keys across seven datasets.
8. Checked referential integrity between orders, customers, products, sellers, and order items.

All primary key and relationship validation checks returned zero errors.

The cleaned datasets are now ready for export and further analysis.

## 12. Exporting Cleaned Datasets

Save the cleaned datasets in the data/processed directory while preserving the original raw CSV files.

In [25]:
# Export all cleaned datasets

for name, df in datasets.items():
    output_file = PROCESSED_PATH / f"{name}_cleaned.csv"
    df.to_csv(output_file, index=False)
    print(f"Saved: {output_file}")

print("\nAll nine cleaned datasets exported successfully!")

Saved: ..\data\processed\customers_cleaned.csv
Saved: ..\data\processed\geolocation_cleaned.csv
Saved: ..\data\processed\order_items_cleaned.csv
Saved: ..\data\processed\order_payments_cleaned.csv
Saved: ..\data\processed\order_reviews_cleaned.csv
Saved: ..\data\processed\orders_cleaned.csv
Saved: ..\data\processed\products_cleaned.csv
Saved: ..\data\processed\sellers_cleaned.csv
Saved: ..\data\processed\category_translation_cleaned.csv

All nine cleaned datasets exported successfully!


In [26]:
# Verify exported datasets

export_summary = []

for name, df in datasets.items():
    file_path = PROCESSED_PATH / f"{name}_cleaned.csv"
    exported_df = pd.read_csv(file_path)

    export_summary.append({
        "Dataset": name,
        "Original Cleaned Rows": len(df),
        "Exported Rows": len(exported_df),
        "Export Status": "Success" if len(df) == len(exported_df) else "Mismatch"
    })

export_df = pd.DataFrame(export_summary)
display(export_df)

,Dataset,Original Cleaned Rows,Exported Rows,Export Status
0,customers,99441,99441,Success
1,geolocation,1000163,1000163,Success
2,order_items,112650,112650,Success
3,order_payments,103886,103886,Success
4,order_reviews,99224,99224,Success
5,orders,99441,99441,Success
6,products,32951,32951,Success
7,sellers,3095,3095,Success
8,category_translation,71,71,Success


In [27]:
# Update dictionary with cleaned geolocation dataset

datasets["geolocation"] = geolocation

print("Updated geolocation dataset:")
print("Rows:", len(datasets["geolocation"]))
print("Duplicates:", datasets["geolocation"].duplicated().sum())

Updated geolocation dataset:
Rows: 738332
Duplicates: 0


In [28]:
for name, df in datasets.items():
    output_file = PROCESSED_PATH / f"{name}_cleaned.csv"
    df.to_csv(output_file, index=False)

print("All nine datasets re-exported successfully!")

All nine datasets re-exported successfully!


## 13. Conclusion

The Olist dataset cleaning and preparation process was completed successfully.

All nine datasets were inspected, cleaned, validated, and exported to the data/processed directory.

A total of 261,831 exact duplicate geolocation records were removed. Missing values were handled according to their business meaning, and date columns were converted to appropriate datetime formats.

Primary key and referential integrity checks returned zero errors. Numerical validation also identified no negative prices, freight charges, payment values, or product weights.

The processed datasets are ready for customer behaviour analysis, segmentation, and retention analysis.